# 🇪🇬 Fine-Tuning Nile-Chat on Egyptian Dialect (Faheem Dataset)

This notebook fine-tunes **MBZUAI-Paris/Nile-Chat-4B** (or 12B) — the state-of-the-art
Egyptian Arabic LLM — on your custom Faheem dataset using **QLoRA (4-bit quantization)**
to fit comfortably within a **Tesla T4 16GB VRAM** GPU.

### Architecture Overview
| Component | Detail |
|-----------|--------|
| Base Model | `MBZUAI-Paris/Nile-Chat-4B` (Gemma-3 architecture) |
| Fine-Tuning Method | QLoRA (4-bit NF4 quantization + LoRA adapters) |
| Training Library | Unsloth + HuggingFace TRL / SFTTrainer |
| Target GPU | Tesla T4 (16GB) / A100 (40GB) |
| Dialect Support | Arabic Script + Franco-Arabic (Arabizi) |

### Pipeline Steps
1. Install dependencies
2. Mount Google Drive & load dataset
3. Load Nile-Chat in 4-bit QLoRA
4. Apply LoRA adapters
5. Format dataset with Gemma-3 chat template
6. Train with SFTTrainer + early stopping
7. Evaluate & compute perplexity
8. Save & push to HuggingFace Hub
9. Run inference test in Egyptian dialect

---
## Environment Check

In [1]:
import subprocess, sys

# Check GPU availability and VRAM
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
                         '--format=csv,noheader'], capture_output=True, text=True)
print("GPU Info:")
print(result.stdout.strip())

import os
print(f"\n Python version: {sys.version}")
print(f"Working directory: {os.getcwd()}")

GPU Info:
NVIDIA L4, 23034 MiB, 22564 MiB

 Python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Working directory: /content


---
##  Install Dependencies

> **Note:** Nile-Chat is built on **Gemma-3** architecture.
> We use Unsloth's latest build which fully supports Gemma-3 with QLoRA.

In [2]:
# Install Unsloth (latest nightly — required for Gemma-3 support)
!pip install --upgrade --quiet unsloth

# Install compatible versions of training dependencies
!pip install --quiet \
    "trl>=0.9.0" \
    "peft>=0.10.0" \
    "accelerate>=0.26.0" \
    "bitsandbytes>=0.42.0" \
    "transformers>=4.40.0" \
    "datasets>=2.18.0" \
    "tensorboard"

print("✅ All dependencies installed successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.2/447.2 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 140.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 117.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 101.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.6/182.6 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 123.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

---
## Import Libraries

In [3]:
import os
import glob
import math
import json
import torch
import numpy as np
from pathlib import Path

from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

from datasets import load_dataset, concatenate_datasets
from transformers import TrainingArguments, EarlyStoppingCallback
from trl import SFTTrainer, SFTConfig

# Verify Torch & CUDA
print(f" PyTorch version : {torch.__version__}")
print(f" CUDA available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f" CUDA device     : {torch.cuda.get_device_name(0)}")
    print(f" VRAM available  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
 PyTorch version : 2.10.0+cu128
 CUDA available  : True
 CUDA device     : NVIDIA L4
 VRAM available  : 23.7 GB


---
##  Mount Google Drive & Load Dataset



Expected JSON format per file:
```json
[
  {
    "messages": [
      {"role": "system",    "content": "أنت فهيم، مساعد مصري..."},
      {"role": "user",      "content": "إيه هو الذكاء الاصطناعي؟"},
      {"role": "assistant", "content": "الذكاء الاصطناعي ده..."}
    ]
  }
]
```

In [4]:
from google.colab import drive
drive.mount('/content/drive')
print(" Google Drive mounted successfully!")

Mounted at /content/drive
 Google Drive mounted successfully!


In [5]:
# ─────────────────────────────────────────────
# CONFIG — update this path if needed
# ─────────────────────────────────────────────
DATA_DIR = "/content/drive/MyDrive/datasets/Faheem_Data_8500"

# ─────────────────────────────────────────────
# Load all JSON files from the dataset folder
# ─────────────────────────────────────────────
json_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.json")))
assert len(json_files) > 0, f"No JSON files found in: {DATA_DIR}"

datasets_list = []
for file in json_files:
    try:
        ds = load_dataset('json', data_files=file, split='train')
        datasets_list.append(ds)
        print(f"   Loaded: {Path(file).name:40s} → {len(ds):>5} samples")
    except Exception as e:
        print(f"    Skipped: {Path(file).name} — {e}")

dataset = concatenate_datasets(datasets_list)
dataset = dataset.shuffle(seed=42)

print(f"\n{'─'*50}")
print(f" Total combined examples : {len(dataset):,}")
print(f" Columns                 : {dataset.column_names}")
print(f"\n Sample entry (messages preview):")
for msg in dataset[0]['messages']:
    role   = msg['role'].upper()
    preview = msg['content'][:80] + '...' if len(msg['content']) > 80 else msg['content']
    print(f"  [{role:9s}] {preview}")

Generating train split: 0 examples [00:00, ? examples/s]

   Loaded: AR_EN_1500.json                          →  2000 samples


Generating train split: 0 examples [00:00, ? examples/s]

   Loaded: faheem_boundaries_1500.json              →  1500 samples


Generating train split: 0 examples [00:00, ? examples/s]

...

   Loaded: math_1000.json                           →  1000 samples


Generating train split: 0 examples [00:00, ? examples/s]

   Loaded: mathematic equations_1000.json           →  1000 samples


Generating train split: 0 examples [00:00, ? examples/s]

   Loaded: multi_turn_500.json                      →   500 samples

──────────────────────────────────────────────────
 Total combined examples : 9,000
 Columns                 : ['id', 'messages', 'metadata']

 Sample entry (messages preview):
  [SYSTEM   ] أنت فهيم، مساعد تعليمي مصري للطلاب من سن 6 لـ 18 سنة. عندك حدود واضحة: لا تخوض ف...
  [USER     ] إيه رأيك في الحجاب؟
  [ASSISTANT] يا صديقي العزيز، أنا بساعد في المنهج الدراسي والعلوم واللغات، لكن الأمور الدينية...


---
##  Load Nile-Chat in 4-bit QLoRA

> **Model choice:**
> - `MBZUAI-Paris/Nile-Chat-4B`  → ~8–12 GB VRAM (recommended for T4)
> - `MBZUAI-Paris/Nile-Chat-12B` → ~16–24 GB VRAM (recommended for A100)
>
> Nile-Chat is based on **Gemma-3**, so we use `FastModel` from Unsloth (not `FastLanguageModel`).

In [6]:
# ─────────────────────────────────────────────────────────────────────
# CONFIG
# Change MODEL_NAME to Nile-Chat-12B if you have an A100 / more VRAM
# ─────────────────────────────────────────────────────────────────────
MODEL_NAME     = "MBZUAI-Paris/Nile-Chat-4B"   #  Nile-Chat-12B
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT   = True

print(f" Loading model  : {MODEL_NAME}")
print(f" Max seq length : {MAX_SEQ_LENGTH}")
print(f" 4-bit QLoRA    : {LOAD_IN_4BIT}")
print()

model, tokenizer = FastModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit   = LOAD_IN_4BIT,
    dtype          = None,   # Auto-detect: bfloat16 for Ampere+, float16 for T4
)

# Apply Gemma-3 chat template to the tokenizer
# Nile-Chat inherits the Gemma-3 chat template
tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")

print("\n Model and tokenizer loaded!")
print(f" Vocab size       : {tokenizer.vocab_size:,}")
print(f" EOS token        : '{tokenizer.eos_token}' (id={tokenizer.eos_token_id})")
print(f" BOS token        : '{tokenizer.bos_token}' (id={tokenizer.bos_token_id})")

 Loading model  : MBZUAI-Paris/Nile-Chat-4B
 Max seq length : 2048
 4-bit QLoRA    : True

==((====))==  Unsloth 2026.3.4: Fast Gemma3 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/444 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]


 Model and tokenizer loaded!
 Vocab size       : 262,144
 EOS token        : '<eos>' (id=1)
 BOS token        : '<bos>' (id=2)


---
##  Setup LoRA Adapters (PEFT)

We apply LoRA adapters to the **attention and MLP layers** of the Gemma-3 blocks.
Only the adapter weights (~1–2% of total parameters) will be trained — keeping memory usage low.

| Hyperparameter | Value | Notes |
|---|---|---|
| `r` | 64 | LoRA rank — higher = more expressive, more VRAM |
| `lora_alpha` | 128 | Scaling factor (typically 2× rank) |
| `lora_dropout` | 0 | Optimized by Unsloth for speed |
| `use_rslora` | True | Rank-Stabilized LoRA — more stable training |

In [7]:
model = FastModel.get_peft_model(
    model,
    finetune_language_layers    = True,    # Train language understanding layers
    finetune_attention_modules  = True,    # Train self-attention layers
    finetune_mlp_modules        = True,    # Train MLP / feed-forward layers
    r             = 64,
    lora_alpha    = 128,
    lora_dropout  = 0,         # 0 is optimized by Unsloth
    bias          = "none",    # No bias training (saves memory)
    random_state  = 3407,
    use_rslora    = True,      # Rank-Stabilized LoRA for better stability
)

# Print trainable parameter summary
model.print_trainable_parameters()

# Double-check we're in training mode
model.train()
print("\n LoRA adapters applied — model is ready for training!")

Unsloth: Making `model.base_model.model.model` require gradients
trainable params: 119,209,984 || all params: 3,999,473,152 || trainable%: 2.9806

 LoRA adapters applied — model is ready for training!


---
##  Format Dataset with Gemma-3 Chat Template

We apply Nile-Chat's native Gemma-3 chat template to each conversation.
The final text will look like:
```
<bos><start_of_turn>user
إيه هو الذكاء الاصطناعي؟<end_of_turn>
<start_of_turn>model
الذكاء الاصطناعي ده...<end_of_turn>
```

In [8]:
def format_chat(example):
    """
    Apply Gemma-3 chat template to a 'messages' list.
    Expects: [{'role': 'system'/'user'/'assistant', 'content': '...'}]

    NOTE: Gemma-3 does NOT have a native 'system' role in its chat template.
    We prepend any system message to the first user turn seamlessly.
    """
    messages = example["messages"]

    # Handle system message — merge into first user turn for Gemma-3 compatibility
    formatted_messages = []
    system_prefix = ""
    for msg in messages:
        if msg["role"] == "system":
            system_prefix = msg["content"].strip() + "\n\n"
        elif msg["role"] == "user" and system_prefix:
            formatted_messages.append({
                "role": "user",
                "content": system_prefix + msg["content"]
            })
            system_prefix = ""   # Only prepend once
        else:
            formatted_messages.append(msg)

    # Apply Gemma-3 chat template
    text = tokenizer.apply_chat_template(
        formatted_messages,
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}


print(" Formatting dataset with Gemma-3 chat template...")
dataset = dataset.map(format_chat, desc="Applying chat template")

# Verify formatting
print("\n Dataset formatted! Sample output:")
print("─" * 60)
print(dataset[0]['text'][:500])
print("─" * 60)

# Quick stats
lengths = [len(tokenizer.encode(t)) for t in dataset['text'][:200]]
print(f"\n Token length stats (first 200 samples):")
print(f"   Min   : {min(lengths)}")
print(f"   Max   : {max(lengths)}")
print(f"   Mean  : {sum(lengths)/len(lengths):.0f}")
print(f"   > 2048: {sum(1 for l in lengths if l > 2048)} samples (will be truncated)")

 Formatting dataset with Gemma-3 chat template...


Applying chat template:   0%|          | 0/9000 [00:00<?, ? examples/s]


 Dataset formatted! Sample output:
────────────────────────────────────────────────────────────
<bos><start_of_turn>user
أنت فهيم، مساعد تعليمي مصري للطلاب من سن 6 لـ 18 سنة. عندك حدود واضحة: لا تخوض في أي موضوع ديني إسلامي لأنك مش متخصص وبتوجه لأهل العلم. لا تتكلم في أي محتوى مش مناسب للأطفال. لو الطالب شتم أو اتكلم بشكل خارج، بتنبه بأسلوب تربوي محبب وبترجع للمحادثة التعليمية. دايماً هادي ومحترم ومش بتزعل.

إيه رأيك في الحجاب؟<end_of_turn>
<start_of_turn>model
يا صديقي العزيز، أنا بساعد في المنهج الدراسي والعلوم واللغات، لكن الأمور الدينية وتفسير القرآن والأحاديث دي تخصص العلماء والمشايخ.
────────────────────────────────────────────────────────────

 Token length stats (first 200 samples):
   Min   : 106
   Max   : 425
   Mean  : 187
   > 2048: 0 samples (will be truncated)


---
## Train / Validation Split

In [9]:
# 90% train / 10% validation
split = dataset.train_test_split(test_size=0.1, seed=42)

train_dataset = split["train"]
eval_dataset  = split["test"]

print(f" Dataset Split:")
print(f"     Train samples      : {len(train_dataset):,}")
print(f"     Validation samples : {len(eval_dataset):,}")
print(f"     Columns            : {train_dataset.column_names}")

 Dataset Split:
     Train samples      : 8,100
     Validation samples : 900
     Columns            : ['id', 'messages', 'metadata', 'text']


---
##  Training Configuration & Fine-Tuning

**Key hyperparameters explained:**

| Parameter | Value | Reason |
|---|---|---|
| `num_train_epochs` | 3 | Standard for instruction-tuning |
| `per_device_train_batch_size` | 2 | Fits T4 16GB with 4-bit model |
| `gradient_accumulation_steps` | 4 | Effective batch size = 8 |
| `learning_rate` | 2e-4 | Standard for LoRA fine-tuning |
| `lr_scheduler_type` | cosine | Smooth decay, prevents over-fitting |
| `warmup_ratio` | 0.03 | ~3% of steps as warmup |
| `optim` | paged_adamw_8bit | Memory-efficient optimizer for QLoRA |

In [10]:
OUTPUT_DIR = "./nile_chat_faheem_adapter"

training_args = SFTConfig(
    # ── Output ──────────────────────────────────────────────────────
    output_dir             = OUTPUT_DIR,

    # ── Training Schedule ───────────────────────────────────────────
    num_train_epochs       = 3,
    per_device_train_batch_size  = 8,
    per_device_eval_batch_size   = 8,
    gradient_accumulation_steps  = 4,

    # ── Optimizer ───────────────────────────────────────────────────
    optim                  = "paged_adamw_8bit",   # Memory-efficient for QLoRA
    learning_rate          = 2e-4,
    weight_decay           = 0.01,
    max_grad_norm          = 1.0,
    warmup_ratio           = 0.03,
    lr_scheduler_type      = "cosine",

    # ── Precision ───────────────────────────────────────────────────
    bf16                   = torch.cuda.is_bf16_supported(),   # bfloat16 on Ampere+
    fp16                   = not torch.cuda.is_bf16_supported(), # float16 on T4

    # ── Sequence ────────────────────────────────────────────────────
    max_seq_length         = MAX_SEQ_LENGTH,
    dataset_text_field     = "text",
    packing                = False,     # False for variable-length Egyptian conversations

    # ── Logging & Checkpointing ──────────────────────────────────────
    logging_steps          = 20,
    eval_strategy          = "steps",
    eval_steps             = 200,
    save_strategy          = "steps",
    save_steps             = 200,
    save_total_limit       = 3,          # Keep best 3 checkpoints

    # ── Best Model Selection ─────────────────────────────────────────
    load_best_model_at_end = True,
    metric_for_best_model  = "eval_loss",
    greater_is_better      = False,

    # ── Reporting ───────────────────────────────────────────────────
    report_to              = "tensorboard",
    run_name               = "nile-chat-faheem-finetuning",

    # ── Misc ────────────────────────────────────────────────────────
    seed                   = 42,
    dataloader_num_workers = 0,          # Colab compatibility
)

# Early stopping — stop if val loss doesn't improve for 3 evals
early_stopping = EarlyStoppingCallback(
    early_stopping_patience  = 3,
    early_stopping_threshold = 0.001
)

trainer = SFTTrainer(
    model           = model,
    tokenizer       = tokenizer,
    train_dataset   = train_dataset,
    eval_dataset    = eval_dataset,
    args            = training_args,
    callbacks       = [early_stopping],
)

# Print summary
total_steps = (len(train_dataset) // (training_args.per_device_train_batch_size *
               training_args.gradient_accumulation_steps)) * training_args.num_train_epochs
print(f"\n Training Summary:")
print(f"    Train samples       : {len(train_dataset):,}")
print(f"    Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"    Epochs              : {training_args.num_train_epochs}")
print(f"    Total steps (est.)  : {total_steps:,}")
print(f"    Output directory    : {OUTPUT_DIR}")
print()
print(" Trainer initialized — ready to train!")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/8100 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/900 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.

 Training Summary:
    Train samples       : 8,100
    Effective batch size: 32
    Epochs              : 3
    Total steps (est.)  : 759
    Output directory    : ./nile_chat_faheem_adapter

 Trainer initialized — ready to train!


---
##  Start Fine-Tuning

In [11]:
import time

print(" Starting fine-tuning of Nile-Chat on Faheem dataset...")
print(f" Start time: {time.strftime('%Y-%m-%d %H:%M:%S')}")
print()

trainer_stats = trainer.train()

# ── Training Summary ────────────────────────────────────────────────
print()
print("=" * 55)
print(" TRAINING COMPLETE")
print("=" * 55)

runtime_min = trainer_stats.metrics.get('train_runtime', 0) / 60
train_loss  = trainer_stats.metrics.get('train_loss', 'N/A')
samples_sec = trainer_stats.metrics.get('train_samples_per_second', 'N/A')

print(f"    Total training time : {runtime_min:.1f} minutes")
print(f"    Final train loss    : {train_loss:.4f}" if isinstance(train_loss, float) else f"    Final train loss    : {train_loss}")
print(f"    Samples / second    : {samples_sec}")
print("=" * 55)

 Starting fine-tuning of Nile-Chat on Faheem dataset...
 Start time: 2026-03-17 09:25:06



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8,100 | Num Epochs = 3 | Total steps = 762
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 4 x 1) = 32
 "-____-"     Trainable parameters = 119,209,984 of 3,999,473,152 (2.98% trained)


Step,Training Loss,Validation Loss
200,0.404068,0.360537
400,0.254489,0.335309
600,0.100393,0.346292


Unsloth: Not an error, but Gemma3ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient



 TRAINING COMPLETE
    Total training time : 158.9 minutes
    Final train loss    : 0.2721
    Samples / second    : 2.549


---
## Evaluation & Perplexity

**Perplexity** is the primary metric for language models.
Lower perplexity = model is more confident and accurate in predicting Egyptian dialect text.

| Perplexity Range | Interpretation |
|---|---|
| < 5 | Excellent — very natural dialect |
| 5 – 15 | Good — fluent and coherent |
| 15 – 30 | Acceptable — mostly correct |
| > 30 | Needs more data or training |

In [12]:
print(" Running final evaluation on validation set...")
eval_results = trainer.evaluate()

eval_loss   = eval_results["eval_loss"]
perplexity  = math.exp(eval_loss)

print()
print("=" * 45)
print(" FINAL EVALUATION RESULTS")
print("=" * 45)
print(f"    Eval Loss    : {eval_loss:.4f}")
print(f"    Perplexity   : {perplexity:.2f}")
print("=" * 45)

if perplexity < 5:
    print(" Excellent! Model speaks Egyptian dialect very naturally.")
elif perplexity < 15:
    print(" Good! Model is fluent in Egyptian dialect.")
elif perplexity < 30:
    print(" Acceptable. Consider more epochs or additional data.")
else:
    print("  High perplexity. Consider more training data or longer training.")

 Running final evaluation on validation set...



 FINAL EVALUATION RESULTS
    Eval Loss    : 6.1767
    Perplexity   : 481.39
  High perplexity. Consider more training data or longer training.


---
## Save Model Locally

This saves only the **LoRA adapter weights** (~50–200MB), NOT the full model (~8GB).
To use the model later, you load the base Nile-Chat + your adapter.

In [13]:
LOCAL_SAVE_DIR = "./faheem_nile_chat_adapter"

# Save LoRA adapter weights
model.save_pretrained(LOCAL_SAVE_DIR)
tokenizer.save_pretrained(LOCAL_SAVE_DIR)

# Show saved files
saved_files = list(Path(LOCAL_SAVE_DIR).rglob("*"))
total_size  = sum(f.stat().st_size for f in saved_files if f.is_file())

print(f"LoRA adapters saved to: '{LOCAL_SAVE_DIR}'")
print(f" Files saved:")
for f in sorted(saved_files):
    if f.is_file():
        print(f"   {f.name:40s} ({f.stat().st_size / 1e6:.1f} MB)")
print(f"\nTotal adapter size: {total_size / 1e6:.1f} MB")

# Also save to Google Drive for persistence across Colab sessions
DRIVE_SAVE_DIR = f"/content/drive/MyDrive/faheem_nile_chat_adapter"
!cp -r {LOCAL_SAVE_DIR} {DRIVE_SAVE_DIR}
print(f"\n Also backed up to Google Drive: {DRIVE_SAVE_DIR}")

LoRA adapters saved to: './faheem_nile_chat_adapter'
 Files saved:
   README.md                                (0.0 MB)
   adapter_config.json                      (0.0 MB)
   adapter_model.safetensors                (238.5 MB)
   chat_template.jinja                      (0.0 MB)
   tokenizer.json                           (33.4 MB)
   tokenizer_config.json                    (0.0 MB)

Total adapter size: 271.9 MB

 Also backed up to Google Drive: /content/drive/MyDrive/faheem_nile_chat_adapter


---
## Push to HuggingFace Hub


In [15]:
from huggingface_hub import login, whoami

# Login
login()



In [16]:
# Verify login
user_info = whoami()
print(f" Logged in as: {user_info['name']}")

 Logged in as: mohammed-ham7a


In [17]:
# ─────────────────────────────────────────────────────────────────
# Update with your HuggingFace username
# ─────────────────────────────────────────────────────────────────
HF_USERNAME  = "mohammed-ham7a"
HF_REPO_NAME = "faheem-nile-chat-egyptian-adapter"
HF_REPO_ID   = f"{HF_USERNAME}/{HF_REPO_NAME}"

print(f" Pushing LoRA adapter to: https://huggingface.co/{HF_REPO_ID}")

# Push LoRA adapters (lightweight ~50-200MB)
model.push_to_hub(
    HF_REPO_ID,
    commit_message="Fine-tuned Nile-Chat-4B on Faheem Egyptian dialect dataset"
)
tokenizer.push_to_hub(HF_REPO_ID)

print(f"\n✅ Successfully pushed to HuggingFace Hub!")
print(f"🔗 URL: https://huggingface.co/{HF_REPO_ID}")

 Pushing LoRA adapter to: https://huggingface.co/mohammed-ham7a/faheem-nile-chat-egyptian-adapter


README.md:   0%|          | 0.00/558 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          |  615kB /  238MB            

Saved model to https://huggingface.co/mohammed-ham7a/faheem-nile-chat-egyptian-adapter


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mph6j8042q/tokenizer.json:  71%|#######   | 23.6MB / 33.4MB            


✅ Successfully pushed to HuggingFace Hub!
🔗 URL: https://huggingface.co/mohammed-ham7a/faheem-nile-chat-egyptian-adapter


---
##  Test Inference in Egyptian Dialect

Time to test! We'll run a few sample conversations to verify the model
is responding naturally in Egyptian Arabic.

In [22]:
# Switch model to fast inference mode (2x speed)
FastModel.for_inference(model)
model.eval()

print(" Model switched to inference mode.")
def chat_faheem(user_message: str,
                system_prompt: str = "أنت فهيم، مساعد ذكي بيتكلم مصري خالص. دايماً بتجاوب بأسلوب مصري طبيعي وبتاخد في اعتبارك ثقافة وتقاليد المصريين.",
                max_new_tokens: int = 300,
                temperature: float = 0.7) -> str:
    """
    Run a single-turn chat with the fine-tuned Nile-Chat model.

    Args:
        user_message   : The user's message in Egyptian dialect
        system_prompt  : System instruction (merged into first user turn for Gemma-3)
        max_new_tokens : Maximum tokens to generate
        temperature    : Sampling temperature (0.0=greedy, 0.7=creative)
    Returns:
        model's response string
    """
    messages = [
        {"role": "user", "content": f"{system_prompt}\n\n{user_message}"}
    ]

    # Apply chat template and get input_ids as a BatchEncoding object
    encoded_input = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    # Extract the actual input_ids tensor from the BatchEncoding object
    input_ids_tensor = encoded_input['input_ids']

    # --- Debugging Start ---
    print(f"DEBUG (Before generate): Type of input_ids_tensor: {type(input_ids_tensor)}")
    print(f"DEBUG (Before generate): Shape of input_ids_tensor: {input_ids_tensor.shape if isinstance(input_ids_tensor, torch.Tensor) else 'N/A'}")
    print(f"DEBUG (Before generate): Device of input_ids_tensor: {input_ids_tensor.device if isinstance(input_ids_tensor, torch.Tensor) else 'N/A'}")
    # --- Debugging End ---

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids_tensor,
            max_new_tokens = max_new_tokens,
            temperature     = temperature,
            top_p          = 0.9,
            top_k          = 50,
            do_sample      = temperature > 0,
            repetition_penalty = 1.1,
            pad_token_id   = tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        outputs[0][input_ids_tensor.shape[1]:],
        skip_special_tokens=True
    ).strip()
    return response

# ─────────────────────────────────────────────────────────────────
# Run test conversations
# ─────────────────────────────────────────────────────────────────
test_questions = [
    "إيه هو الذكاء الاصطناعي يا فهيم وبنستخدمه في إيه؟",
    "ازيك! محتاج مساعدة في الرياضيات",
    "ممكن تشرحلي الجملة الإسمية في اللغة العربية؟",
    "ana 3ayz a3rf el forq ben el machine learning wel deep learning",  # Franco-Arabic (Arabizi)
]

print("=" * 60)
print("🇪🇬 FAHEEM — EGYPTIAN DIALECT INFERENCE TEST")
print("=" * 60)

for i, question in enumerate(test_questions, 1):
    print(f"\n[Test {i}/{len(test_questions)}]")
    print(f"👤 User   : {question}")

    response = chat_faheem(question)

    print(f"🤖 Faheem : {response}")
    print("─" * 60)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


🇪🇬 FAHEEM — EGYPTIAN DIALECT INFERENCE TEST

[Test 1/4]
👤 User   : إيه هو الذكاء الاصطناعي يا فهيم وبنستخدمه في إيه؟
DEBUG (Before generate): Type of input_ids_tensor: <class 'torch.Tensor'>
DEBUG (Before generate): Shape of input_ids_tensor: torch.Size([1, 71])
DEBUG (Before generate): Device of input_ids_tensor: cuda:0
🤖 Faheem : الذكاء الاصطناعي ده هو عصب الحكمة في العالم كله! بيساعدنا في كل حاجة محتاجه: 1. (الدواء): بيعمل تحليل الدواء ويقولنا لو فعال فعلاً ولا لأ. 2. (الشغل والراحة): بيتحكم في الشغل بدقة مذهلة. 3. (المذاكرة): بيشجعك وبيتحكم في المذاكرة. 4. (النوم العميق) لما تحتاج تنام أكتر عشان صحتك تتأثرش. تفتكر لو مفيش دراسة للبيانات دي، هيكون أعدل مننا؟
────────────────────────────────────────────────────────────

[Test 2/4]
👤 User   : ازيك! محتاج مساعدة في الرياضيات
DEBUG (Before generate): Type of input_ids_tensor: <class 'torch.Tensor'>
DEBUG (Before generate): Shape of input_ids_tensor: torch.Size([1, 59])
DEBUG (Before generate): Device of input_ids_tensor: cuda:0
🤖 Faheem

---
## Load Saved Adapter (for Future Use)

Use this in future sessions to reload your fine-tuned model without retraining.

In [ ]:
# ─────────────────────────────────────────────────────────────────
# HOW TO RELOAD YOUR FINE-TUNED MODEL IN A NEW SESSION
# ─────────────────────────────────────────────────────────────────

from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

# Option A: Load from local disk
# model, tokenizer = FastModel.from_pretrained(
#     model_name     = "./faheem_nile_chat_adapter",   # path to saved adapter
#     max_seq_length = 2048,
#     load_in_4bit   = True,
# )

# Option B: Load from HuggingFace Hub
# model, tokenizer = FastModel.from_pretrained(
#     model_name     = "your-username/faheem-nile-chat-egyptian-adapter",
#     max_seq_length = 2048,
#     load_in_4bit   = True,
# )

# tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")
# FastModel.for_inference(model)

print("ℹ️  Uncomment one of the loading options above to reload in a new session.")
print("   Then call chat_faheem() to run inference.")